In [ ]:
# Llama-3.2 is gated. Store a Kaggle secret named HF_TOKEN.
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    # Outside Kaggle: fall back to the HF_TOKEN env var or a cached `huggingface-cli login`.
    if os.environ.get('HF_TOKEN'):
        login(token=os.environ['HF_TOKEN'])
        print('Hugging Face login from HF_TOKEN env var: OK')
    else:
        print('HF login was not completed automatically:', exc)
        print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""


In [ ]:
from pathlib import Path

LAYERS = [0, 7, 15]
ENERGY_THRESHOLD = 50.0   # minimum cumulative energy (%) to retain
N_PLOT = 200              # visualize only the first 200 singular values

OUT_DIR = Path(os.environ.get('A1_RESULT_ROOT', '/kaggle/working/Assignment1')) / 'q4'
OUT_DIR.mkdir(parents=True, exist_ok=True)

svd_results = {}
for k in LAYERS:
    # Analyze the weights of the query projection layer (q_proj) of layer k
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

    # Full SVD: W = U diag(S) Vh, singular values S are sorted in descending order
    U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

    # Cumulative energy (%) retained by the top-r singular values, r = 1..len(S)
    energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100

    # Minimum target rank r such that cumulative energy >= 50%
    min_rank = int(torch.searchsorted(energy_ratios, torch.tensor(ENERGY_THRESHOLD)).item()) + 1
    assert energy_ratios[min_rank - 1] >= ENERGY_THRESHOLD
    assert min_rank == 1 or energy_ratios[min_rank - 2] < ENERGY_THRESHOLD

    svd_results[k] = {'S': S, 'energy': energy_ratios, 'min_rank': min_rank}
    print(
        f'Layer {k:2d}: q_proj shape={tuple(q_weight.shape)}, full rank={len(S)}, '
        f'sigma_1={S[0]:.3f}, sigma_16={S[15]:.3f}, sigma_200={S[199]:.3f}, sigma_min={S[-1]:.4f}, '
        f'energy@16={energy_ratios[15]:.2f}%, energy@200={energy_ratios[199]:.2f}%, '
        f'min rank for >=50% energy={min_rank}'
    )


In [ ]:
import pandas as pd

min_rank_table = pd.DataFrame(
    {f'Layer {k}': [svd_results[k]['min_rank']] for k in LAYERS},
    index=['Minimum target rank for query projection weights (>= 50% energy)'],
)
min_rank_table.to_csv(OUT_DIR / 'q4_min_rank.csv')

summary = pd.DataFrame([
    {
        'layer': k,
        'full_rank': len(r['S']),
        'min_rank_50pct_energy': r['min_rank'],
        'min_rank_fraction_of_full': r['min_rank'] / len(r['S']),
        'sigma_1': r['S'][0].item(),
        'sigma_200': r['S'][199].item(),
        'sigma_1_over_sigma_200': (r['S'][0] / r['S'][199]).item(),
        'energy_top1_pct': r['energy'][0].item(),
        'energy_top16_pct': r['energy'][15].item(),
        'energy_top200_pct': r['energy'][199].item(),
    }
    for k, r in svd_results.items()
])
summary.to_csv(OUT_DIR / 'q4_svd_summary.csv', index=False)
display(min_rank_table)
display(summary)


In [ ]:
# 6 charts: for each layer, cumulative energy (line) and singular value magnitudes (bar)
idx = torch.arange(1, N_PLOT + 1).numpy()
BLUE, GRID = '#2a78d6', '#e4e3df'
fig, axes = plt.subplots(2, len(LAYERS), figsize=(12, 6.6))

for col, k in enumerate(LAYERS):
    S = svd_results[k]['S'][:N_PLOT].numpy()
    energy = svd_results[k]['energy'][:N_PLOT].numpy()
    min_rank = svd_results[k]['min_rank']

    ax = axes[0, col]
    ax.plot(idx, energy, color=BLUE, lw=2)
    if min_rank <= N_PLOT:
        ax.axhline(ENERGY_THRESHOLD, color='gray', ls='--', lw=1)
        ax.axvline(min_rank, color='#eb6834', ls='--', lw=1, label=f'min rank = {min_rank}')
        ax.legend(loc='lower right')
    ax.set_title(f'Layer {k} q_proj: Cumulative Energy')
    ax.set_xlabel('Target Rank')
    ax.set_ylabel('Cumulative Energy (%)')
    ax.set_axisbelow(True)
    ax.grid(True, color=GRID)

    ax = axes[1, col]
    ax.bar(idx, S, width=1.0, color=BLUE)
    ax.set_title(f'Layer {k} q_proj: Singular Value Magnitude')
    ax.set_xlabel('Singular Value Index')
    ax.set_ylabel('Singular Value Magnitude')
    ax.set_axisbelow(True)
    ax.grid(True, color=GRID)

fig.tight_layout()
fig.savefig(OUT_DIR / 'q4_svd_all_layers.png', dpi=200)
plt.show()

# Also save each of the 6 charts individually
for k in LAYERS:
    S = svd_results[k]['S'][:N_PLOT].numpy()
    energy = svd_results[k]['energy'][:N_PLOT].numpy()
    for kind in ['energy', 'magnitude']:
        f, ax = plt.subplots(figsize=(6, 4))
        if kind == 'energy':
            ax.plot(idx, energy, color=BLUE, lw=2)
            ax.set_xlabel('Target Rank'); ax.set_ylabel('Cumulative Energy (%)')
        else:
            ax.bar(idx, S, width=1.0, color=BLUE)
            ax.set_xlabel('Singular Value Index'); ax.set_ylabel('Singular Value Magnitude')
        ax.set_title(f'Layer {k} q_proj')
        ax.set_axisbelow(True)
        ax.grid(True, color=GRID)
        f.tight_layout()
        f.savefig(OUT_DIR / f'q4_layer{k}_{kind}.png', dpi=200)
        plt.close(f)
print('Saved figures to', OUT_DIR)
